# 30장 · LangGraph의 검색 노드와 답변 노드

검색 노드는 질문에서 관련 문서를 찾고, 답변 노드는 그 문서를 읽습니다. 두 노드가 공유할 상태에 본문과 출처를 함께 담습니다.

검색 노드가 돌려준 한 문서에는 본문·ID·자료명·쪽 번호가 함께 들어 있습니다. 답변 노드는 본문을 읽고, 마지막 실습에서는 상태에 남은 자료명과 쪽 번호로 인용을 표시합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## retrieve()의 문서와 출처 반환

노트북의 자료는 앞서 사용한 가상 도서관 안내 세 쪽입니다. 읽는 법에 집중하려고 로컬 검색 함수는 세 낱말의 유무를 거리로 바꾸며 k개를 고릅니다. Chroma 서버나 외부 임베딩을 호출하는 셀은 없습니다.

### 셀 01 · retrieve()의 문서 메타데이터

대출 질문을 넣고 목록의 첫 번째 레코드를 읽으세요.

In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

print(retrieve('대출 기간은?', 1)[0])

{'id': 'library-1', 'text': '도서 대출 기간은 14일입니다.', 'source': '가상 도서관 이용 안내', 'page': 1}


library-1에 14일이라는 문장, 가상 도서관 이용 안내, 1쪽이 함께 있습니다. text만 남기면 다음 노드에서 출처를 복원할 수 없습니다.

본문만 복사해 다음 노드에 넘기면 ‘14일’이라는 답은 쓸 수 있습니다. 하지만 답변 끝에 어느 문서 몇 쪽인지 붙이려면 다시 검색해야 합니다. 검색을 한 번만 하고도 인용하려면 첫 반환값의 text, id, page를 한 레코드로 넘기세요.

### retrieve(question, k)의 반환값

<p>검색에 쓰는 세 낱말은 대출·휴관·반납입니다. 하나도 들어 있지 않으면 빈 리스트를 돌려줍니다. 일치하는 낱말이 있으면 거리와 문서 ID로 정렬한 뒤 앞의 k개를 반환합니다. 셀 01은 k=1의 첫 레코드를 출력합니다.</p>

### 문장에 출처와 쪽 번호를 함께 보관하기

<p>26장에서는 documents의 원소가 문자열이었습니다. 여기서는 한 문서를 id·text·source·page 네 필드로 받습니다. 답변에는 text를 쓰고, 어떤 자료의 몇 쪽인지 확인할 때는 같은 레코드에서 source와 page를 읽습니다.</p>

## StateGraph에 검색·답변 노드 연결

검색 결과를 상태에 넣을 때는 documents 키에 목록을 그대로 반환합니다. 다음 노드는 같은 목록에서 답을 꺼냅니다. 별도로 다시 검색하면 처음 확인한 근거와 다른 문서를 답에 쓰게 될 수 있습니다.

### 셀 02 · 검색·답변 노드를 간선으로 연결

아래 그래프에서 START 다음에 연결한 노드를 찾고 휴관 질문을 실행하세요.

등록한 노드 이름은 search와 compose이고 실제 함수 이름은 search_node와 answer_node입니다. 휴관 질문에서는 최종 상태의 answer에 안내 문장이, 첫 문서의 page에 쪽 번호가 남습니다.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class State(TypedDict):
    question: str
    documents: list[dict]
    answer: str

def search_node(state):
    return {"documents": retrieve(state["question"], k=1)}

def answer_node(state):
    docs = state["documents"]
    return {"answer": docs[0]["text"] if docs else "제공된 자료에서 확인할 수 없습니다."}

builder = StateGraph(State)
builder.add_node("search", search_node)
builder.add_node("compose", answer_node)
builder.add_edge(START, "search")
builder.add_edge("search", "compose")
builder.add_edge("compose", END)
graph = builder.compile()

request = {'question': '휴관 요일은 언제인가요?', 'documents': [], 'answer': ''}
result = graph.invoke(request)
print(result['answer'], result['documents'][0]['page'])

도서관은 월요일에 휴관합니다. 2


월요일이라는 답 뒤에 2가 붙습니다. 검색 함수가 고른 2쪽을 compose가 그대로 읽었습니다. 답 옆의 숫자는 결과 순위가 아닌 원문의 쪽수입니다. result는 실행 결과 전체의 변수 이름입니다. 답은 result["answer"], 원문 쪽수는 result["documents"][0]["page"]에서 읽습니다.

#### 검색 함수의 반환값을 그래프에 넘길 때

<mark>검색 노드가 남긴 레코드의 page는 답변 노드가 answer를 쓴 뒤에도 읽을 수 있습니다.</mark> 최종 답변 문자열만 보면 휴관 안내가 어느 쪽에서 왔는지 사라집니다. 최종 상태의 documents에서 첫 문서를 꺼내 출력의 쪽 번호와 짝지으세요.

선수 개념 · 26장 LangGraph StateGraph의 상태와 노드

#### 근거 레코드를 넘기는 고정 간선

```mermaid
flowchart TD
s["START"] --> r["search"]
r -->|documents| c["compose"]
c -->|answer| e["END"]
```

search는 documents를 쓰고 compose는 그 목록을 읽어 answer를 씁니다. 빈 목록을 반환해도 고정 간선은 compose로 이어집니다.

### 과제 03 · 빈 검색 결과의 응답 분기

‘주차 요금은?’을 넣고 documents와 answer를 차례로 확인하세요.

In [3]:
# 초기 상태의 질문을 바꾸어 그래프를 실행하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>empty = graph.invoke({**request, &#x27;question&#x27;: &#x27;주차 요금은?&#x27;})
print(empty[&#x27;documents&#x27;], empty[&#x27;answer&#x27;])</code></pre>
<p>실행 결과</p><pre><code>[] 제공된 자료에서 확인할 수 없습니다.</code></pre>
<p>빈 목록과 확인할 수 없다는 안내가 나옵니다. 세 낱말 모두 질문에 없어 검색 함수가 빈 목록을 돌려준 경우입니다. 연결 오류에 같은 안내를 쓰면 장애가 가려집니다.</p>
</details>

### 검색에 성공했지만 근거가 없을 때

<p>주차 질문에는 자료의 세 낱말이 없습니다. search가 빈 리스트를 기록해도 고정 간선은 compose로 이어집니다. compose가 빈 리스트를 읽고 안내를 씁니다. 다음 장에서는 검색 함수가 예외를 낸 경우와 비교합니다.</p>

## 돌아보기

휴관 질문에 대해 마지막 셀이 받은 값은 답변과 2쪽이라는 출처입니다. 새 질문에도 이 둘이 함께 표시되도록, 이미 찾은 문서에서 자료명과 쪽을 읽는 함수를 작성하세요.

### 과제 04 · 검색을 반복하지 않고 답변에 출처 달기

반납함 질문을 기존 그래프에 한 번 보내세요. 최종 상태만 받는 render_answer를 작성해 답과 자료명·쪽을 표시합니다. 근거가 비어 있으면 안내 문장만 반환합니다.

In [ ]:
request = {"question": "반납함은 몇 시간 이용하나요?", "documents": [], "answer": ""}
# render_answer는 retrieve를 다시 호출하지 않고 상태에 남은 출처를 사용합니다.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>request = {"question": "반납함은 몇 시간 이용하나요?", "documents": [], "answer": ""}

def render_answer(state):
    if not state["documents"]:
        return state["answer"]
    doc = state["documents"][0]
    return f"{state['answer']} [{doc['source']}, {doc['page']}쪽]"

print(render_answer(graph.invoke(request)))</code></pre>
<p>실행 결과</p><pre><code>무인 반납함은 24시간 이용할 수 있습니다. [가상 도서관 이용 안내, 3쪽]</code></pre>
<p>근거를 찾은 search와 답을 만든 compose를 거친 뒤에만 출처를 표시합니다. 문서가 비면 첫 원소를 꺼내지 않아 주차 질문에도 같은 함수를 쓸 수 있습니다.</p>
</details>

## 검색과 답변에서 서로 다른 문서가 쓰이지 않게 연결하기

검색 결과에 근거 ID가 들어 있어도 답변을 만들 때 문서를 다시 찾으면 검색 시점의 자료와 어긋날 수 있습니다. 이번에는 검색을 한 번만 하고 그 결과를 다음 노드로 전달합니다.

### 셀 05 · 한 질문에 검색 노드가 몇 번 불렸는지 세기

함수 바깥에 이 실행만 기록할 목록을 두고 검색 노드가 받을 질문을 남깁니다. 검색 노드를 직접 한 번 실행한 뒤 문서 ID를 확인하세요.

In [4]:
calls_seen = []
def measured_search(state):
    calls_seen.append(state["question"])
    return search_node(state)

trace_input = {**request, "question": "무인 반납함 이용 시간은?"}
delta = measured_search(trace_input)
print(calls_seen, delta["documents"][0]["id"])

['무인 반납함 이용 시간은?'] library-3


질문 하나와 library-3이 나옵니다. 다음 노드에 이미 필요한 레코드가 들어 있습니다. 표시용 출처를 얻기 위해 retrieve를 한 번 더 부를 필요가 없습니다.

### 과제 06 · 검색 1회로 본문과 쪽수까지 반환하는 그래프 만들기

measured_search와 새 인용 노드를 연결하세요. 인용 노드는 documents의 첫 레코드를 사용하고 빈 목록이면 안내를 돌려줍니다. 호출 기록을 비운 다음 한 질문을 보내 검색 횟수와 결과를 확인하세요.

In [5]:
# citation_node를 정의하고 search에서 citation으로 이어지는 그래프를 만드세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def citation_node(state):
    docs = state[&quot;documents&quot;]
    if not docs:
        return {&quot;answer&quot;: &quot;확인할 자료가 없습니다.&quot;}
    first_doc = docs[0]
    return {&quot;answer&quot;: f&quot;{first_doc[&#x27;text&#x27;]} [{first_doc[&#x27;id&#x27;]}, {first_doc[&#x27;page&#x27;]}쪽]&quot;}

cited_builder = StateGraph(State)
cited_builder.add_node(&quot;search&quot;, measured_search)
cited_builder.add_node(&quot;cite&quot;, citation_node)
cited_builder.add_edge(START, &quot;search&quot;)
cited_builder.add_edge(&quot;search&quot;, &quot;cite&quot;)
cited_builder.add_edge(&quot;cite&quot;, END)
calls_seen.clear()
cited_result = cited_builder.compile().invoke(trace_input)
print(len(calls_seen), cited_result[&quot;answer&quot;])</code></pre>
<p>실행 결과</p><pre><code>1 무인 반납함은 24시간 이용할 수 있습니다. [library-3, 3쪽]</code></pre>
<p>출력 앞의 1은 검색 호출 수입니다. 뒤에는 반납함의 24시간 안내와 library-3, 3쪽이 붙습니다. 세 값은 같은 문서 레코드에서 읽었습니다.</p>
</details>

<mark>답변 문장과 인용 위치를 같은 검색 결과에서 꺼내세요.</mark> 검색이 한 번인데 화면의 문장이 3쪽, 출처가 2쪽이면 두 값을 조립한 코드를 확인할 수 있습니다.

다음 장은 검색이 성공해도 답변 보류가 필요한 경우를 구분합니다.